In [0]:
-- Create and activate your schema 
CREATE SCHEMA IF NOT EXISTS brightcoffee 
    COMMENT 'Bright Coffee Shop analytics schema';
  
USE brightcoffee;
 
 SELECT current_database();
  

 

-- ───01) dim_customers─────────────────────────────── 
CREATE TABLE IF NOT EXISTS dim_customers ( 
    customer_id     BIGINT      NOT NULL, 
    first_name      STRING      NOT NULL, 
    last_name       STRING      NOT NULL, 
    email           STRING, 
    tier            STRING      DEFAULT 'Bronze', 
    loyalty_points  INT         DEFAULT 0, 
    created_at      DATE, 
    CONSTRAINT pk_customer PRIMARY KEY (customer_id)
    ) 
USING DELTA 
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');
 

-- ───3) dim_products-----  
CREATE TABLE IF NOT EXISTS dim_products ( 
    product_id      BIGINT          NOT NULL, 
    product_code    STRING          NOT NULL, 
    product_name    STRING          NOT NULL, 
    price           DECIMAL(8,2)    NOT NULL, 
    category        STRING          DEFAULT 'Uncategorized', 
    is_active       BOOLEAN         DEFAULT true, 
    CONSTRAINT pk_product PRIMARY KEY (product_id) 
) 
USING DELTA 
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');
 

_______3) FACT_ORDERS

CREATE TABLE IF NOT EXISTS fact_orders ( 
    order_id        BIGINT          NOT NULL, 
    customer_id     BIGINT          NOT NULL, 
    product_id      BIGINT          NOT NULL, 
    store_id        BIGINT          NOT NULL, 
    order_date      DATE            NOT NULL, 
    quantity        INT             NOT NULL, 
    unit_price      DECIMAL(8,2)    NOT NULL, 
    total_amount    DECIMAL(10,2)   NOT NULL, 
    CONSTRAINT pk_order PRIMARY KEY (order_id) 
) 
USING DELTA 
PARTITIONED BY (order_date);


-----------04) Enable column mapping on dim_customers, then rename loyalty_points to points----

ALTER TABLE dim_customers 
    SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name','delta.minReaderVersion'   
= '2','delta.minWriterVersion'   = '5');
 

-- ─── rename and drop  ────────────────────────────────────── 
ALTER TABLE dim_customers RENAME COLUMN loyalty_points TO points;

ALTER TABLE dim_customers DROP COLUMN loyalty_points;


----Insert at least 5 customers into dim_customers---

INSERT INTO dim_customers (customer_id,first_name,last_name,email,tier) 
VALUES 
(101,'Amu','Nition','amu101@gmail.com','Gold'), 
(102,'John','Blake','John104@webmail.com','Bronze'), 
(103,'Thandi','Thobana','thandi111@exmail.com','Silver'),
(104,'Zakes','Bantwini','Zakes666@webmail.com','Gold'),
(105,'Black','Diemane','Black777@hotmail.com','Silver');


----6) Insert 5 products

INSERT INTO dim_products(product_id,product_code,product_name,price)
VALUES
(111,'Beer','Black_Label',50),
(112,'Cider','Savanah',60),
(113,'Draught','Windoek',70),
(114,'Whiskey','Jameson',25),
(115,'Bourbon','Jim Bean', 26);


---Update all bronze-tier customers----

SELECT customer_id, first_name, tier, points
FROM   dim_customers 
WHERE  tier = 'Bronze';
 

UPDATE dim_customers 
SET    points = points + 50 
WHERE  tier = 'Bronze';


SELECT customer_id, first_name, tier, points 
FROM   dim_customers 
WHERE  tier = 'Bronze';


-----08) Back up dim_customers----

-- Step 1: back up first 
CREATE TABLE dim_customers_backup 
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
AS SELECT * FROM dim_customers;
 
  
SELECT * FROM dim_customers WHERE customer_id = 101;
  
DELETE FROM dim_customers WHERE customer_id = 101;
 
   
SELECT * FROM dim_customers;
 
 
DESCRIBE HISTORY dim_customers;


